# Epoch Spring Camp 2026 - Take Home Task 3

In this task, you will build and compare two recommender system models:

- **Matrix Factorization (MF)** using a dot product of embeddings  
- **Neural Collaborative Filtering (MLP-based)** using a multi-layer perceptron  

You are provided with:
- Preprocessed interaction data
- Evaluation pipeline

You are expected to implement:
- Negative sampling
- Model architectures
- Training loop

---

The purpose of this task is to understand how neural networks can model user–item interactions beyond simple similarity.

## Imports

In [7]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import random
import pandas as pd
import numpy as np

In [15]:
from google.colab import drive
drive.mount('/content/drive')



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Loading Data

We begin by loading the interaction data.

In [19]:
df = pd.read_csv("/interactions.csv")  # columns: user_id, movie_id

print("Num users:", df['user_id'].nunique())
print("Num items:", df['item_id'].nunique())
print("Num interactions:", len(df))

Num users: 942
Num items: 1447
Num interactions: 55375


Each row represents a **positive interaction** (implicit feedback).

## Train / Test Split

We split the data into training and testing sets.

The model will learn from the training data and be evaluated on unseen interactions in the test set.

In [20]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

## Negative Sampling

The dataset only contains **positive interactions** (user interacted with item).

However, to train a model, we also need **negative examples** (user did not interact with item).

---

### Why do we need this?

We treat recommendation as a **binary classification problem**:

- Positive interaction → label = 1  
- No interaction → label = 0  

Since we don’t have explicit negatives, we **sample them**.

---

### Your Task

For each `(user, item)` interaction:
- Keep it as a **positive sample (label = 1)**
- Sample one or more items the user has **not interacted with**
  - Add them as **negative samples (label = 0)**

---

### Hints

- You can randomly sample items
- Make sure sampled negatives are **not already in the dataset**
- You may use a `set` of `(user, item)` pairs for fast lookup

---

Implement a function:
`sample_negatives(df, num_neg=1)`
that returns a dataframe with columns:
`[user, item, label]`

In [21]:
def sample_negatives(df, num_neg=1):
    # TODO:
    # 1. Create a set of (user, item) interactions
    # 2. For each positive interaction:
    #    - add (user, item, 1)
    #    - sample 'num_neg' negative items
    # 3. Return a dataframe with columns: user, item, label
    pos_interactions = set(zip(df['user_id'], df['item_id']))
    all_items = df['item_id'].unique()
    data = []
    for u , i in pos_interactions:

      user_positives = {item for user, item in pos_interactions if user == u}
      potential_negatives = list(set(all_items) - user_positives)
      count=0
      neg_items = random.sample(potential_negatives, k=min(num_neg, len(potential_negatives)))
      while count<len(neg_items):
        for neg_item in neg_items:
          data.append([u, neg_item, 0])
          count+=1
    return pd.DataFrame(data, columns=['user', 'item', 'label'])

## PyTorch Dataset

We now wrap our processed data into a PyTorch `Dataset`.

This allows us to:
- Access individual samples as `(user, item, label)`
- Easily plug into a `DataLoader` for batching

You do not need to modify this part.

In [22]:
class InteractionDataset(Dataset):
    def __init__(self, df):
        self.users = torch.tensor(df['user'].values)
        self.items = torch.tensor(df['item'].values)
        self.labels = torch.tensor(df['label'].values).float()

    def __len__(self):
        return len(self.users)

    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.labels[idx]

## DataLoader

The `DataLoader` will:
- Batch the data
- Shuffle the training data
- Feed it to the model during training

In [23]:
train_data = sample_negatives(train_df)

train_loader = DataLoader(InteractionDataset(train_data), batch_size=256, shuffle=True)

Why are we sampling negatives only for the training data?

## Quick Exploration

Before building models, take a moment to explore the data.

Try to understand:
- How many interactions each user has
- How popular certain items are

This can give intuition about the dataset.

In [24]:
# Interactions per user
user_counts = df['user_id'].value_counts()
print(user_counts.describe())

# Interactions per item
item_counts = df['item_id'].value_counts()
print(item_counts.describe())

count    942.000000
mean      58.784501
std       54.696664
min        3.000000
25%       19.000000
50%       39.500000
75%       80.750000
max      378.000000
Name: count, dtype: float64
count    1447.000000
mean       38.268832
std        57.956847
min         1.000000
25%         3.000000
50%        13.000000
75%        47.500000
max       501.000000
Name: count, dtype: float64


## Baseline Model: Matrix Factorization (MF)

We represent:
- Each **user** as a vector (embedding)
- Each **item** as a vector (embedding)

To predict interaction:
- Compute the **dot product** between user and item embeddings

This gives a **score** indicating how likely the user is to interact with the item.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Computes their dot product as the output score

In [25]:
class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()


        # TODO:
        # - Define user embedding layer
        # - Define item embedding layer
        self.user_embedding = nn.Embedding(num_embeddings=num_users, embedding_dim=emb_dim)
        self.item_embedding=nn.Embedding(num_embeddings=num_items,embedding_dim=emb_dim)


    def forward(self, user, item):
        # TODO:
        # - Get user and item embeddings
        # - Compute dot product
        # - Return a single score per pair
        user_emb=self.user_embedding(user)
        item_emb=self.item_embedding(item)
        score=(user_emb*item_emb).sum(dim=1)
        return score

## Training the MF Model

Now train your Matrix Factorization model.

You will need to:
- Define a loss function
- Define an optimizer
- Iterate over the DataLoader
- Update model parameters

---

### Hints

- Use **Binary Cross Entropy loss**
- Apply **sigmoid** to model outputs if needed
- Typical steps:
  - forward pass
  - compute loss
  - backward pass
  - optimizer step

In [26]:
def train(model, dataloader, epochs):
    # TODO:
    # - Move model to device (optional)
    # - Define optimizer
    # - Define loss function
    emb_dim=16
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    optimizer=torch.optim.Adam(model.parameters(),lr=0.01)
    criterion = nn.BCEWithLogitsLoss()
    losses = []  # track loss per epoch

    for epoch in range(epochs):
        total_loss = 0

        for user, item, label in dataloader:
            # TODO:
            # - Forward pass
            # - Compute loss
            # - Backpropagation
            # - Optimizer step
            user, item, label = user.to(device), item.to(device), label.to(device).float()
            #forward pass
            optimizer.zero_grad()
            outputs=model(user,item)
            # - Compute loss
            loss=criterion(outputs,label)
            # - Backpropagation
            loss.backward()
            # - Optimizer step
            optimizer.step()
            total_loss+=loss.item()
            pass

        losses.append(total_loss)
        print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

    return losses

## Evaluation (Hit@K)

We evaluate the model using a ranking-based metric.

For each user:
- Take one positive item
- Sample multiple negative items
- Rank all items using the model
- Check if the positive item is in the top-K

This is called **Hit@K**.

In [2]:

def hit_at_k(model, test_df, full_df, K=10, num_neg=100):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()
    model.to(device)

    hits = 0
    total = len(df) # We evaluate every positive interaction in the test set

    # Map interactions for quick lookup
    interacted_items = full_df.groupby('user_id')['item_id'].apply(set).to_dict()
    all_items = full_df['item_id'].unique()

    with torch.no_grad(): # Disable gradient calculation for speed/memory
        for _, row in df.iterrows():
            u = int(row['user_id'])
            pos_item = int(row['item_id'])

            # 1. Sample Negatives
            negatives = []
            while len(negatives) < num_neg:
                neg_item = np.random.choice(all_items)
                if neg_item not in interacted_items.get(u, set()):
                    negatives.append(neg_item)

            # 2. Prepare Tensors
            # We need a list of the 1 positive + 100 negatives
            item_list = [pos_item] + negatives
            user_tensor = torch.tensor([u] * (num_neg + 1)).to(device)
            item_tensor = torch.tensor(item_list).to(device)

            # 3. Get Scores
            scores = model(user_tensor, item_tensor)

            # 4. Rank and Check Hit
            # We want to see if the item at index 0 (the positive) is in the top K
            # torch.topk returns values and indices of the highest scores
            _, top_indices = torch.topk(scores, K)

            top_indices = top_indices.cpu().numpy()
            if 0 in top_indices:
                hits += 1

    return hits / total

full_df = df

# TODO:
# 1. Initialize MF model
num_users = full_df['user_id'].nunique()
num_items = full_df['item_id'].nunique()
model=MF(num_users, num_items, emb_dim=16)

# 2. Train it
train(model, train_loader, epochs=5)

# 3. Evaluate it
score = hit_at_k(model, test_df, full_df, K=10)

NameError: name 'df' is not defined

If your score is low, try playing with the hyperparameters before moving on! Try sampling more negatives, or playing with the embedding dimensions.

Conversely, if your score is high, play with the values of K or number of negatives in evaluation (dec K, inc negatives).

## Neural Model: MLP-Based Recommender

Instead of using a dot product, we can learn a more flexible interaction function using a neural network.

Approach:
- Learn user and item embeddings
- **Concatenate** them
- Pass through a **Multi-Layer Perceptron (MLP)**

This allows the model to capture more complex relationships than simple similarity.

---

### Your Task

Implement a model that:
1. Learns user and item embeddings
2. Concatenates them
3. Passes them through an MLP to produce a score

The choice of activation functions is upto you.

In [ ]:
class MLP(nn.Module):
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()

        # TODO:
        # - Define user embedding layer
        self.user_embedding = nn.Embedding(num_users, emb_dim)

        # - Define item embedding layer
        self.item_embedding = nn.Embedding(num_items, emb_dim)

        # - Define MLP layers
        self.mlp = nn.Sequential(
            nn.Linear(emb_dim * 2, 64),
            nn.ReLU(),

            nn.Linear(64, 32),
            nn.ReLU(),

            nn.Linear(32, 1)
        )

    def forward(self, user, item):
        # TODO:
        # - Get embeddings
        user_emb = self.user_embedding(user)
        item_emb = self.item_embedding(item)

        # - Concatenate user and item embeddings
        x = torch.cat([user_emb, item_emb], dim=1)

        # - Pass through MLP
        out = self.mlp(x)

        # - Return a single score
        return out.squeeze()

## Train and Evaluate the MLP Model

Repeat the same steps as before:
- Train the model
- Evaluate on the test set

Compare its performance with the MF model.

In [ ]:
# TODO:
# 1. Initialize MLP model
num_users = full_df['user_id'].nunique()
num_items = full_df['item_id'].nunique()

model = MLP(num_users, num_items, emb_dim=16)

# 2. Train it
train(model, train_loader, epochs=5)

# 3. Evaluate it
score = hit_at_k(model, test_df, full_df, K=10)

print(f"Hit@10: {score:.4f}")

## Comparison & Analysis

You have now implemented:
- Matrix Factorization (MF)
- MLP-based recommender

---

### Compare the Models

- What score did each model achieve?
- Which model performed better?

---

### Think & Reflect

- Why might the MLP model outperform MF?
- In what cases might MF perform just as well or better?
- How does embedding size affect performance?
- Did you observe any signs of overfitting?

---

### Some Experiments

If you want to explore further:
- Try different embedding dimensions
- Change number of MLP layers
- Try different activation functions

---

## Bonus Task: Neural Collaborative Filtering (NCF)

In this task, you implemented:
- Matrix Factorization (MF)
- MLP-based recommender

The paper [Neural Collaborative Filtering](https://arxiv.org/pdf/1708.05031) proposes combining both ideas.

---

### Idea

- MF captures **linear interactions**
- MLP captures **nonlinear interactions**

NCF combines both by:
1. Computing MF output
2. Computing MLP output
3. Combining them into a final prediction

---

### Your Task

Design a model that:
- Uses both MF and MLP components
- Combines their outputs
- Trains end-to-end

---

### Hints

- You can:
  - Concatenate MF and MLP representations
  - Or combine their final scores
- Think about:
  - Should embeddings be shared or separate?
  - How to balance both components?

---

Does combining both approaches improve performance over MF and MLP individually?